In [1]:
!pip install -q streamlit plotly pandas numpy scikit-learn pyngrok rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 60.8 MB/s eta 0:00:00


In [2]:
!npm install localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 4s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

In [35]:
%%writefile app.py
import streamlit as st
import plotly.express as px
import pandas as pd
import numpy as np
import rasterio
import glob
import os
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

st.set_page_config(layout="wide", page_title="Dashboard Cianobacterias")

# ---------- Palette ----------
PALETTE = {
    'Amatitlán': 'rgb(46,80,144)',   # Azul Oscuro
    'Atitlán':  'rgb(74,123,183)',  # Azul Medio
    'Bajo':     'rgb(150,180,220)',  # Azul claro para bajo
    'Medio':    'rgb(255,140,66)',   # Naranja
    'Alto':     'rgb(214,69,69)'     # Rojo
}

# ---------- Header / descripción ----------
st.title("Monitoreo de Cianobacterias — Lagos Amatitlán y Atitlán")
st.markdown(
    """
**Descripción:** Este dashboard carga GeoTIFFs por fecha desde las carpetas
`NCDI_AMATITLAN` y `NCDI_ATITLAN`, calcula estadísticos por raster (media, mediana, std, min, max)
y permite explorar patrones, definir niveles de riesgo y entrenar modelos básicos.

**Flujo recomendado:** 1) Selecciona transformación (percentile stretch / log / gain).
2) Define nivel de riesgo.
3) Usa filtros en la barra lateral para enlazar las visualizaciones.
"""
)

# ---------- Sidebar: transform + parametros ----------
st.sidebar.markdown("### Transformación / inspección de TIFFs")
transform_method = st.sidebar.selectbox("Transformación a aplicar antes de estadísticos",
                                        options=["Ninguna", "Percentile stretch", "Log (log1p)", "Multiplicar (gain)"],
                                        index=1)
p_low = st.sidebar.slider("Percentil bajo (%)", 0.0, 10.0, 1.0, step=0.5)
p_high = st.sidebar.slider("Percentil alto (%)", 90.0, 100.0, 99.0, step=0.5)
gain = st.sidebar.number_input("Gain multiplicativo ", value=1.0, step=0.1, format="%.2f")

# opción para usar media normalizada en visualizaciones
usar_normalizada = st.sidebar.checkbox("Usar media normalizada (0-1) en gráficas", value=True)

# ---------- Función de carga/estadísticos con transform ----------
@st.cache_data
def load_raster_stats_with_transform(folder_path, lake_name, transform_method, p_low, p_high, gain):
    rows = []
    pattern = os.path.join(folder_path, "*.tif")
    import re
    for fp in sorted(glob.glob(pattern)):
        try:
            with rasterio.open(fp) as src:
                dtype = src.dtypes[0] if src.count>0 else None
                band = src.read(1, masked=True)
                # calcular cantidad de válidos
                if hasattr(band, "mask"):
                    valid_mask = ~band.mask
                    valid_count = int(valid_mask.sum())
                    total_count = int(band.size)
                else:
                    valid_count = int(np.count_nonzero(~np.isnan(band)))
                    total_count = int(band.size)
                valid_pct = 0.0 if total_count == 0 else float(valid_count) / float(total_count) * 100.0

                if valid_count == 0:
                    rows.append({'lago': lake_name, 'filepath': fp, 'fecha': None, 'mes': None,
                                 'mean': np.nan, 'median': np.nan, 'std': np.nan, 'min': np.nan, 'max': np.nan,
                                 'valid_pct': valid_pct, 'dtype': dtype})
                    continue

                data_vals = band.compressed() if hasattr(band, "compressed") else band[~np.isnan(band)]
                vals = data_vals.astype(float)

                # aplicar transformaciones
                if transform_method == "Percentile stretch":
                    lo = np.percentile(vals, p_low)
                    hi = np.percentile(vals, p_high)
                    if hi - lo <= 0:
                        stretched = vals - lo
                    else:
                        stretched = (vals - lo) / (hi - lo)
                    stretched = np.clip(stretched, 0, 1)
                    vals = stretched
                elif transform_method == "Log (log1p)":
                    # si hay negativos, desplazar antes de log (si corresponde)
                    if np.any(vals < 0):
                        vals = vals - vals.min() + 1e-6
                    vals = np.log1p(vals)
                elif transform_method == "Multiplicar (gain)":
                    vals = vals * float(gain)

                mean = float(np.mean(vals))
                median = float(np.median(vals))
                std = float(np.std(vals))
                mn = float(np.min(vals))
                mx = float(np.max(vals))

                # parse fecha desde nombre archivo
                fecha = None
                m = re.search(r"(\d{4}-\d{2}-\d{2})", os.path.basename(fp))
                if m:
                    fecha = datetime.strptime(m.group(1), "%Y-%m-%d").date()
                else:
                    fecha = datetime.fromtimestamp(os.path.getmtime(fp)).date()

                rows.append({
                    'lago': lake_name,
                    'filepath': fp,
                    'fecha': fecha,
                    'mes': fecha.month if fecha else None,
                    'mean': mean,
                    'median': median,
                    'std': std,
                    'min': mn,
                    'max': mx,
                    'valid_pct': valid_pct,
                    'dtype': dtype
                })
        except Exception as e:
            # registrar fila de fallo para trazabilidad
            rows.append({'lago': lake_name, 'filepath': fp, 'fecha': None, 'mes': None,
                         'mean': np.nan, 'median': np.nan, 'std': np.nan, 'min': np.nan, 'max': np.nan,
                         'valid_pct': 0.0, 'dtype': None})
            continue
    if not rows:
        return pd.DataFrame(columns=['lago','filepath','fecha','mes','mean','median','std','min','max','valid_pct','dtype'])
    return pd.DataFrame(rows)

# rutas (ajusta si están en otro lado)
path_amatitlan = "NCDI_AMATITLAN"
path_atitlan = "NCDI_ATITLAN"

# cargar
df_ami = load_raster_stats_with_transform(path_amatitlan, "Amatitlán", transform_method, p_low, p_high, gain)
df_ati = load_raster_stats_with_transform(path_atitlan, "Atitlán", transform_method, p_low, p_high, gain)
data = pd.concat([df_ami, df_ati], ignore_index=True)

# limpieza mínima: separar filas sin fecha (informa al usuario)
missing_dates = data['fecha'].isna().sum()
if missing_dates > 0:
    st.warning(f"Se detectaron {missing_dates} archivos con fecha no extraíble; se excluirán de análisis temporal y del modelado.")
data = data.dropna(subset=['fecha']).copy()
data['mes'] = data['fecha'].apply(lambda d: d.month)

# Normalización simple para visualizaciones
if 'mean' in data.columns and not data['mean'].isna().all():
    mean_min = float(data['mean'].min())
    mean_max = float(data['mean'].max())
    if mean_max - mean_min > 0:
        data['mean_norm'] = (data['mean'] - mean_min) / (mean_max - mean_min)
    else:
        data['mean_norm'] = 0.0
else:
    data['mean_norm'] = np.nan

# información resumida al usuario
st.markdown("#### Resumen de datos cargados")
st.write(f"Archivos Amatitlán: {len(df_ami)}, Archivos Atitlán: {len(df_ati)}, Registros con fecha válida: {len(data)}")
st.markdown("**Nota:** `valid_pct` muestra porcentaje de pixeles válidos  por raster. Si es bajo, la imagen puede no contener señal útil.")

st.markdown("### Inspección rápida de TIFFs")
st.dataframe(data[['lago','filepath','valid_pct','dtype','min','max','mean']].sort_values('filepath'), use_container_width=True)

# ---------- Definición de nivel de riesgo (sidebar) ----------
st.sidebar.markdown("### Definir nivel de riesgo")
metodo_riesgo = st.sidebar.radio("Método para definir niveles de riesgo",
                                 options=["Umbrales fijos", "Por cuantiles", "Personalizado"])

if metodo_riesgo == "Umbrales fijos (por defecto)":
    bins = [-1, 30, 60, 1e9]
    labels = ['Bajo', 'Medio', 'Alto']
    data['nivel_riesgo'] = pd.cut(data['mean'], bins=bins, labels=labels)
elif metodo_riesgo == "Por cuantiles (tertiles)":
    q1 = data['mean'].quantile(1/3)
    q2 = data['mean'].quantile(2/3)
    bins = [-1, q1, q2, 1e9]
    labels = ['Bajo', 'Medio', 'Alto']
    data['nivel_riesgo'] = pd.cut(data['mean'], bins=bins, labels=labels)
else:
    st.sidebar.markdown("Ajusta umbrales (valores en la escala de 'mean')")
    min_mean = float(data['mean'].min())
    max_mean = float(data['mean'].max())
    b1 = st.sidebar.slider("Límite Bajo→Medio", min_mean, max_mean, float(min_mean + (max_mean-min_mean)/3))
    b2 = st.sidebar.slider("Límite Medio→Alto", min_mean, max_mean, float(min_mean + 2*(max_mean-min_mean)/3))
    if b2 <= b1:
        st.sidebar.warning("El límite Medio→Alto debe ser mayor que Bajo→Medio. Ajustando automáticamente.")
        b2 = b1 + 1e-6
    bins = [-1, b1, b2, 1e9]
    labels = ['Bajo', 'Medio', 'Alto']
    data['nivel_riesgo'] = pd.cut(data['mean'], bins=bins, labels=labels)

data['nivel_riesgo'] = data['nivel_riesgo'].astype(str)
st.sidebar.markdown("**Distribución resultante por nivel de riesgo**")
st.sidebar.write(data['nivel_riesgo'].value_counts())

# ---------- Sidebar filtros  ----------
st.sidebar.header("Filtros")
lago_seleccionado = st.sidebar.multiselect("Seleccionar Lagos", options=sorted(data['lago'].unique()), default=sorted(data['lago'].unique()))
mes_min = int(data['mes'].min())
mes_max = int(data['mes'].max())
mes_seleccionado = st.sidebar.slider("Rango de Meses", 1, 12, (mes_min, mes_max))
riesgo_options = sorted(data['nivel_riesgo'].unique())
riesgo_filtrado = st.sidebar.multiselect("Filtrar por nivel de riesgo", options=riesgo_options, default=riesgo_options)
fecha_opciones = sorted(data['fecha'].astype(str).unique())
fecha_seleccionada = st.sidebar.selectbox("Fecha para detalle", options=["Todas"] + fecha_opciones, index=0)

# aplicar filtros
mask = (data['lago'].isin(lago_seleccionado)) & \
       (data['mes'].between(mes_seleccionado[0], mes_seleccionado[1])) & \
       (data['nivel_riesgo'].isin(riesgo_filtrado))
if fecha_seleccionada != "Todas":
    mask = mask & (data['fecha'].astype(str) == fecha_seleccionada)

data_filtrada = data[mask].copy()

st.markdown(f"**Registros mostrados:** {len(data_filtrada)}")

# ---------- Visualizaciones (Mínimo 8 diferentes) ----------
col1, col2 = st.columns(2)

with col1:
    st.subheader("Distribución de Media NCDI por Lago")
    y_field = 'mean_norm' if usar_normalizada else 'mean'
    fig1 = px.box(data_filtrada, x='lago', y=y_field, color='lago',
                  color_discrete_map={'Amatitlán': PALETTE['Amatitlán'], 'Atitlán': PALETTE['Atitlán']})
    fig1.update_layout(showlegend=False, yaxis_title="Media NCDI (normalizada)" if usar_normalizada else "Media NCDI")
    st.plotly_chart(fig1, use_container_width=True)

with col2:
    st.subheader("Proporción de Niveles de Riesgo")
    riesgo_counts = data_filtrada['nivel_riesgo'].value_counts().reindex(['Bajo','Medio','Alto']).fillna(0)
    colors_pie = [PALETTE['Bajo'], PALETTE['Medio'], PALETTE['Alto']]
    fig2 = px.pie(values=riesgo_counts.values, names=riesgo_counts.index, hole=0.3, color_discrete_sequence=colors_pie)
    st.plotly_chart(fig2, use_container_width=True)

col3, col4 = st.columns(2)
with col3:
    st.subheader("Media vs Std (scatter) - tamaño = rango (max-min)")
    data_filtrada = data_filtrada.copy()
    data_filtrada['range'] = data_filtrada['max'] - data_filtrada['min']
    fig3 = px.scatter(data_filtrada, x='std', y=y_field, color='lago', size='range',
                      hover_data=['fecha','filepath','min','max','valid_pct'],
                      color_discrete_map={'Amatitlán': PALETTE['Amatitlán'], 'Atitlán': PALETTE['Atitlán']},
                      labels={'std':'Desviación', y_field:'Media NCDI'})
    st.plotly_chart(fig3, use_container_width=True)

with col4:
    st.subheader("Variación Temporal de la Media")
    mes_agg = data_filtrada.groupby(['fecha','lago'])[y_field].mean().reset_index().sort_values('fecha')
    fig4 = px.line(mes_agg, x='fecha', y=y_field, color='lago', markers=True,
                   color_discrete_map={'Amatitlán': PALETTE['Amatitlán'], 'Atitlán': PALETTE['Atitlán']})
    fig4.update_layout(yaxis_title="Media NCDI (normalizada)" if usar_normalizada else "Media NCDI")
    st.plotly_chart(fig4, use_container_width=True)

st.subheader("Análisis adicional")
r1, r2 = st.columns(2)
with r1:
    st.markdown("### Histograma de medias (por lago)")
    fig_hist = px.histogram(data_filtrada, x='mean' if not usar_normalizada else 'mean_norm', nbins=30, color='lago',
                            marginal='box', color_discrete_map={'Amatitlán': PALETTE['Amatitlán'], 'Atitlán': PALETTE['Atitlán']})
    fig_hist.update_layout(xaxis_title="Media NCDI (normalizada)" if usar_normalizada else "Media NCDI")
    st.plotly_chart(fig_hist, use_container_width=True)

with r2:
    st.markdown("### Heatmap: media por mes y lago")
    heat = data_filtrada.groupby(['mes','lago'])['mean'].mean().reset_index()
    if heat.empty:
        st.info("No hay datos suficientes para generar el heatmap (filtra menos).")
    else:
        heat_pivot = heat.pivot(index='mes', columns='lago', values='mean').fillna(0)
        fig_heat = px.imshow(heat_pivot, labels=dict(x="Lago", y="Mes", color="Media NCDI"),
                             x=heat_pivot.columns, y=heat_pivot.index)
        st.plotly_chart(fig_heat, use_container_width=True)

# Tabla de detalle
st.markdown("### Tabla de registros y detalle")
st.dataframe(data_filtrada.sort_values(['fecha','lago']), use_container_width=True)

# ---------- Modelos predictivos  ----------
st.subheader("Comparación de Modelos Predictivos")

# Preparar dataset para modelos: usar data (no solo filtrada) para tener más muestras,
# pero mostrar conteo de clases en la vista filtrada.
df_model = data.copy()
df_model = df_model.dropna(subset=['mean','std','min','max','median'])
df_model = pd.get_dummies(df_model, columns=['mes'], prefix='mes', drop_first=True)

X_cols = [c for c in df_model.columns if c in ['mean','std','min','max','median'] or c.startswith('mes_')]
X = df_model[X_cols].values
y = df_model['nivel_riesgo'].values

import collections
class_counts_total = collections.Counter(y)
st.write("Distribución de clases (dataset completo):", dict(class_counts_total))

# comprobar diversidad de clases
if len([c for c in class_counts_total.values() if c>0]) < 2:
    st.error("No hay suficiente diversidad de clases en el dataset completo para entrenar modelos. "
             "Prueba: usar 'Por cuantiles (tertiles)' para definir niveles de riesgo o ajustar transformación.")
else:
    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    from sklearn.model_selection import train_test_split
    def try_safe_split(X, y, attempts=10, test_size=0.25, random_state=42):
        for attempt in range(attempts):
            try:
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size,
                                                                    random_state=random_state+attempt,
                                                                    stratify=y if len(set(y))>1 else None)
            except ValueError:
                # si stratify falla por alguna razón, intentar sin stratify
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size,
                                                                    random_state=random_state+attempt,
                                                                    stratify=None)
            if len(set(y_train)) > 1:
                return X_train, X_test, y_train, y_test
        # fallback: intentar con test_size reducido
        for attempt in range(attempts):
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=max(0.1, test_size/2),
                                                                random_state=random_state+100+attempt,
                                                                stratify=None)
            if len(set(y_train)) > 1 and len(set(y_test)) > 0:
                return X_train, X_test, y_train, y_test
        return None, None, None, None

    X_train, X_test, y_train, y_test = try_safe_split(X, y_enc, attempts=12, test_size=0.25, random_state=42)

    if X_train is None:
        st.error("No fue posible crear un conjunto de entrenamiento con más de una clase. Intenta reducir filtros o cambiar método de riesgo.")
    else:
        modelos = {
            'Random Forest': RandomForestClassifier(random_state=42),
            'Regresión Logística': LogisticRegression(max_iter=2000, random_state=42),
            'SVM': SVC(probability=True, random_state=42)
        }

        modelos_seleccionados = st.multiselect("Seleccionar modelos a comparar", list(modelos.keys()), default=list(modelos.keys()))
        if not modelos_seleccionados:
            st.info("Seleccione al menos un modelo para ver resultados.")
        else:
            resultados = []
            cm_figs = {}
            for nombre in modelos_seleccionados:
                m = modelos[nombre]
                try:
                    m.fit(X_train, y_train)
                    y_pred = m.predict(X_test)
                    acc = accuracy_score(y_test, y_pred)
                    resultados.append({'Modelo': nombre, 'Accuracy': acc})

                    cm = confusion_matrix(y_test, y_pred, labels=range(len(le.classes_)))
                    cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
                    cm_fig = px.imshow(cm_df, text_auto=True, labels=dict(x="Predicho", y="Verdadero", color="Count"),
                                       title=f"Matriz: {nombre}")
                    cm_figs[nombre] = cm_fig
                except ValueError as e:
                    st.warning(f"No se pudo entrenar {nombre}: {e}")
                except Exception as e:
                    st.warning(f"Error entrenando {nombre}: {e}")

            if resultados:
                df_resultados = pd.DataFrame(resultados).sort_values('Accuracy', ascending=False)

                colA, colB = st.columns([1,2])

                with colA:
                    st.markdown("#### Desempeño (tabla)")
                    st.dataframe(df_resultados, use_container_width=True)

                with colB:
                    st.markdown("#### Desempeño (barra)")
                    color_seq = [PALETTE['Amatitlán'], PALETTE['Atitlán'], PALETTE['Medio']]
                    fig_bar = px.bar(df_resultados, x='Modelo', y='Accuracy', color='Modelo',
                                     color_discrete_sequence=color_seq)
                    fig_bar.update_layout(showlegend=False, yaxis=dict(range=[0,1]))
                    st.plotly_chart(fig_bar, use_container_width=True)

                for nombre, fig_cm in cm_figs.items():
                    st.plotly_chart(fig_cm, use_container_width=True)
            else:
                st.info("Ningún modelo produjo resultados (posible causa: clases insuficientes o error durante entrenamiento).")

# ---------- Export ----------
st.markdown("### Exportar datos procesados")
csv = data.to_csv(index=False)
st.download_button("Descargar CSV de estadísticos", data=csv, file_name="estadisticos_ncdi.csv", mime="text/csv")


Overwriting app.py


In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
!streamlit run app.py &>/content/logs.txt &

In [38]:
!pip install -q pyngrok

import os, time
from urllib.parse import urlparse
from pyngrok import ngrok, conf

NGROK_TOKEN = "34oe6fCuTeiUfdpOgqmJGC2aA52_33sGHP1FVz4tgvN7g87JE"   # <- deben sacar un auth token en ngrok y se coloca aqui, luego se abre la url que da ngrok y se puede visualizar

conf.get_default().auth_token = NGROK_TOKEN

tunnel = ngrok.connect(addr="8501", proto="http", bind_tls=True)
public_url = tunnel.public_url if hasattr(tunnel, "public_url") else str(tunnel)
print("Public URL:", public_url)

u = urlparse(public_url)
host = u.hostname
port = 443 if u.scheme == "https" else 80
print("Host:", host, "Port:", port)

cfg_dir = ".streamlit"
os.makedirs(cfg_dir, exist_ok=True)
cfg = f"""
[server]
headless = true
address = "0.0.0.0"
port = 8501
enableCORS = false
enableXsrfProtection = false

[browser]
# host público (sin https://)
serverAddress = "{host}"
serverPort = {port}
"""
with open(os.path.join(cfg_dir, "config.toml"), "w") as f:
    f.write(cfg)

print("Wrote .streamlit/config.toml with public host. Restarting Streamlit...")

os.system("pkill -f streamlit || true")
time.sleep(1)

os.system("nohup streamlit run app.py &> /content/logs.txt &")

print("Streamlit iniciado. Espera unos segundos y abre:", public_url)
print("Puedes ver logs con: tail -n 200 /content/logs.txt")

Public URL: https://nonexaggeratory-nonlitigiously-carissa.ngrok-free.dev
Host: nonexaggeratory-nonlitigiously-carissa.ngrok-free.dev Port: 443
Wrote .streamlit/config.toml with public host. Restarting Streamlit...
Streamlit iniciado. Espera unos segundos y abre: https://nonexaggeratory-nonlitigiously-carissa.ngrok-free.dev
Puedes ver logs con: tail -n 200 /content/logs.txt


In [33]:
import os

print("Deteniendo procesos de Streamlit...")
os.system("pkill -f streamlit || true")
print("Deteniendo procesos de ngrok...")
os.system("pkill -f ngrok || true")
print("Procesos de Streamlit y ngrok detenidos. Intenta volver a iniciar tu aplicación.")

Deteniendo procesos de Streamlit...
Deteniendo procesos de ngrok...
Procesos de Streamlit y ngrok detenidos. Intenta volver a iniciar tu aplicación.
